In [220]:
import os
import PyPDF2
import numpy as np
import pandas as pd
from tabula.io import read_pdf
from datetime import datetime
import re

In [221]:
file_name = r"C:\Users\admin\Downloads\20.11.2023 £1,796.40 On-Line Auto Sport.pdf"

r"C:\Users\admin\Downloads\20.02.2023 £3,285.15 On-Line Auto Sport Limited.pdf"

'C:\\Users\\admin\\Downloads\\20.02.2023 £3,285.15 On-Line Auto Sport Limited.pdf'

In [222]:
invoice_type = "Products"

input_file = fr"C:\Users\admin\Downloads\11.03.2024 £4,382.98 On-Line Auto Sport.pdf"

In [223]:
name = "On-Line Auto Sport"

table1 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(160,310,280,580),
                  columns=[580],
                  pandas_options={'header': None},
                  encoding="windows-1254")

heading = table1[0]
display(heading)

docnum = heading[0][0].split(' ',1)[1].replace(' ','')
print(docnum)

date = heading[0][1].split(':')[1].replace(' ','')
date = str(datetime.strptime(date, "%d/%m/%Y"))
print(date)

ordernum = heading[0][2].split(':')[1].replace(' ','')

transfernum = None
print(ordernum)
print(transfernum)


,0
0,Invoice 212510
1,Date : 11/03/2024
2,Your Order No. :PO33922/PO34413
3,Account Ref. : MLPE01


212510
2024-03-11 00:00:00
PO33922/PO34413
None


In [224]:
with open(input_file,'rb') as pdf_file:
    pdf_reader = PyPDF2.PdfReader(pdf_file)
    num_pages = len(pdf_reader.pages)

print(num_pages)

2


In [225]:
#create loop to go through the pages
all_content = []
for page in range(1, num_pages + 1):
    table2 = read_pdf(input_file,
                    pages=page,
                    silent=True,
                    guess=False,
                    area=(277,24,720,577),
                    columns=[83.5,154,438,502,577],
                    pandas_options={'header': None},
                    encoding="windows-1254")
    contenti = table2[0]
    all_content.append(contenti)

content = pd.concat(all_content).reset_index(drop=True)
display(content)

,0,1,2,3,4
0,Quantity,Part Number,Product Description,Each To,tal Amount
1,NaN,M,** ORDER No. PO33922 **,NaN,NaN
2,1.00,R00049084,BMW Z3 FRONT LIP SPOILER,£91.21,£91.21
3,1.00,R00088028,VW GOLF MK7 REAR DIFFUSER GLASS BLACK,£134.77,£134.77
4,1.00,R00025025,MB W202 C CLASS 93-97 REAR LOWER APRON,£96.66,£96.66
5,1.00,R00025070,MERCEDES W202 RIEGER FRONT BUMPER,£223.94,£223.94
6,1.00,R00053467,BMW 3 SERIES F30/F31 REAR DIFFUSER 335/340,£134.77,£134.77
7,1.00,R00051002,ASTRA F RADIATOR GRILLE,£53.77,£53.77
8,1.00,R00055103,AUDI TT 8N SIDE SKIRT LEFT,£67.39,£67.39
9,1.00,R00055104,AUDI TT 8N SIDE SKIRT RIGHT,£67.39,£67.39


In [226]:
content[0] = pd.to_numeric(content[0], errors='coerce')  # Remove anything that not number in column 0
content = content.dropna(subset=[0]).reset_index(drop=True) # Remove rows with NaN in column 0

content[[3,4]] = content[[3,4]].replace('[£,$, ]','', regex=True).astype('string')

content

,0,1,2,3,4
0,1.0,R00049084,BMW Z3 FRONT LIP SPOILER,91.21,91.21
1,1.0,R00088028,VW GOLF MK7 REAR DIFFUSER GLASS BLACK,134.77,134.77
2,1.0,R00025025,MB W202 C CLASS 93-97 REAR LOWER APRON,96.66,96.66
3,1.0,R00025070,MERCEDES W202 RIEGER FRONT BUMPER,223.94,223.94
4,1.0,R00053467,BMW 3 SERIES F30/F31 REAR DIFFUSER 335/340,134.77,134.77
5,1.0,R00051002,ASTRA F RADIATOR GRILLE,53.77,53.77
6,1.0,R00055103,AUDI TT 8N SIDE SKIRT LEFT,67.39,67.39
7,1.0,R00055104,AUDI TT 8N SIDE SKIRT RIGHT,67.39,67.39
8,1.0,R00025020,MB W202 C CLASS RERA WINDOW SPOILER,46.97,46.97
9,1.0,R00025045,MB W201 190 SIDE SKIRT LEFT,46.97,46.97


In [227]:
content.rename(columns={
    0: 'Quantity',
    1: 'Part Number',
    2: 'Product Description',
    3: 'Each',
    4: 'Total Amount'}, inplace=True)

display(content)

,Quantity,Part Number,Product Description,Each,Total Amount
0,1.0,R00049084,BMW Z3 FRONT LIP SPOILER,91.21,91.21
1,1.0,R00088028,VW GOLF MK7 REAR DIFFUSER GLASS BLACK,134.77,134.77
2,1.0,R00025025,MB W202 C CLASS 93-97 REAR LOWER APRON,96.66,96.66
3,1.0,R00025070,MERCEDES W202 RIEGER FRONT BUMPER,223.94,223.94
4,1.0,R00053467,BMW 3 SERIES F30/F31 REAR DIFFUSER 335/340,134.77,134.77
5,1.0,R00051002,ASTRA F RADIATOR GRILLE,53.77,53.77
6,1.0,R00055103,AUDI TT 8N SIDE SKIRT LEFT,67.39,67.39
7,1.0,R00055104,AUDI TT 8N SIDE SKIRT RIGHT,67.39,67.39
8,1.0,R00025020,MB W202 C CLASS RERA WINDOW SPOILER,46.97,46.97
9,1.0,R00025045,MB W201 190 SIDE SKIRT LEFT,46.97,46.97


In [228]:
dict_content = content.to_dict(orient='records')
dict_content

line_items=[]
for item in dict_content:

    partNum = item['Part Number']
    desc = item['Product Description']
    quantity = item['Quantity']
    netTotal = item['Total Amount']

    print(partNum)
    
    line_item = {"line_type": "inventory",
                        "sku": str(partNum),
                        "name": desc,
                        "quantity": int(quantity),
                        "net_total": float(netTotal),
                        "tax_type": "INPUT2"}
    
    line_items.append(line_item)
    
print(line_items)

R00049084
R00088028
R00025025
R00025070
R00053467
R00051002
R00055103
R00055104
R00025020
R00025045
R00025046
R00025073
R00025074
R00048221
R00055072
R00055113
R00055114
R00055488
R00055489
R00057004
R00057005
R00059012
R00059017
R00088247
R00099844
R00099858
R00088172
R00059524
R00059328
R00034134
R00034135
R00088095
R00047006
R00047007
R00034177
R00051119
R00027014
[{'line_type': 'inventory', 'sku': 'R00049084', 'name': 'BMW Z3 FRONT LIP SPOILER', 'quantity': 1, 'net_total': 91.21, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'R00088028', 'name': 'VW GOLF MK7 REAR DIFFUSER GLASS BLACK', 'quantity': 1, 'net_total': 134.77, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'R00025025', 'name': 'MB W202 C CLASS 93-97 REAR LOWER APRON', 'quantity': 1, 'net_total': 96.66, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'R00025070', 'name': 'MERCEDES W202 RIEGER FRONT BUMPER', 'quantity': 1, 'net_total': 223.94, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 

In [229]:
#create loop to go through the pages
all_total_content = []

for page in range(1, num_pages + 1):
    table3 = read_pdf(input_file,
                    pages=page,
                    silent=True,
                    guess=False,
                    area=(700,320,820,575),
                    columns=[460,575],
                    pandas_options={'header': None},
                    encoding='windows-1254')
    
    if table3 != []:
        total_contenti=table3[0]
        all_total_content.append(total_contenti)

total_content = pd.concat(all_total_content).reset_index(drop=True)
display(total_content)


,0,1
0,Net Amount,"£3,652.52"
1,Carriage,£0.00
2,Total Net,"£3,652.52"
3,VAT Amount,£730.46
4,Invoice Total,"£4,382.98"


In [230]:
row_index = total_content.index[total_content[0] == "Invoice Total"].tolist()[0]

final_total = float(str(total_content[1][row_index]).replace('£','').replace(',',''))
display(final_total)

row_shipping = total_content.index[total_content[0] == "Carriage"].tolist()[0]
shipping = float(str(total_content[1][row_shipping]).replace('£','').replace(',',''))
display(shipping)


4382.98

0.0

In [231]:
#Adding shipping to the line_items
if shipping != 0:
    line_shipping = {"line_type":"shipping_expense",
                "sku": None,
                "name": "shipping",
                "Quantity": int(1),
                "net_total": float(shipping),
                "tax_type": "INPUT2"}

    line_items.append(line_shipping)
print(line_items)

[{'line_type': 'inventory', 'sku': 'R00049084', 'name': 'BMW Z3 FRONT LIP SPOILER', 'quantity': 1, 'net_total': 91.21, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'R00088028', 'name': 'VW GOLF MK7 REAR DIFFUSER GLASS BLACK', 'quantity': 1, 'net_total': 134.77, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'R00025025', 'name': 'MB W202 C CLASS 93-97 REAR LOWER APRON', 'quantity': 1, 'net_total': 96.66, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'R00025070', 'name': 'MERCEDES W202 RIEGER FRONT BUMPER', 'quantity': 1, 'net_total': 223.94, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'R00053467', 'name': 'BMW 3 SERIES F30/F31 REAR DIFFUSER 335/340', 'quantity': 1, 'net_total': 134.77, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'R00051002', 'name': 'ASTRA F RADIATOR GRILLE', 'quantity': 1, 'net_total': 53.77, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'R00055103', 'name': 'AUDI TT 8N SIDE SKIRT LEFT', 'quantity': 

In [232]:
payload = {}
keys = ["Source File",
        "Type",
        "Name",
        "Date",
        "Reference No.",
        "Order No.",
        "Transfer No.",
        "Document No.",
        "Line Items",
        "Total"]

values = [file_name,
        invoice_type,
        name,
        date,
        docnum,
        ordernum,
        transfernum,
        None,
        line_items,
        final_total]

for i, key in enumerate(keys):
    payload[key] = values[i]

payload

{'Source File': 'C:\\Users\\admin\\Downloads\\20.11.2023 £1,796.40 On-Line Auto Sport.pdf',
 'Type': 'Products',
 'Name': 'On-Line Auto Sport',
 'Date': '2024-03-11 00:00:00',
 'Reference No.': '212510',
 'Order No.': 'PO33922/PO34413',
 'Transfer No.': None,
 'Document No.': None,
 'Line Items': [{'line_type': 'inventory',
   'sku': 'R00049084',
   'name': 'BMW Z3 FRONT LIP SPOILER',
   'quantity': 1,
   'net_total': 91.21,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': 'R00088028',
   'name': 'VW GOLF MK7 REAR DIFFUSER GLASS BLACK',
   'quantity': 1,
   'net_total': 134.77,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': 'R00025025',
   'name': 'MB W202 C CLASS 93-97 REAR LOWER APRON',
   'quantity': 1,
   'net_total': 96.66,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': 'R00025070',
   'name': 'MERCEDES W202 RIEGER FRONT BUMPER',
   'quantity': 1,
   'net_total': 223.94,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   '